# 00 — Setup, public data, and supported-SQL compiler

**Run this notebook first.** It creates a mode-specific experiment
directory, downloads or reads the public datasets, generates
deterministic SQL assets, compiles only sound relations from a
declared SQL fragment, and validates compiled relations against an
independent construction oracle plus executed bag-semantics checks.

The certified fragment covers single-table row-set queries with
half-open numeric or categorical partitions, plus additive
`COUNT`/`SUM` aggregates. Joins, `AVG`, `DISTINCT`, `GROUP BY`,
`ORDER BY`, and `LIMIT` are intentionally fail-closed. This notebook
does **not** claim arbitrary SQL derivability.


## Beginner-safe execution

This version has one executable cell. Select **Runtime -> Run all**
once. Do not copy or run individual fragments. The cell creates and
verifies all companion modules before importing `arbiter_data`.


In [1]:
print("\n=== Notebook 00 stage 1/7: install dependencies ===")
import subprocess
import sys

# Install only dependencies that are absent or version-sensitive in Colab.
# The final environment manifest records every resolved version.
packages = [
    "duckdb==1.3.2",
    "osqp==1.0.4",
    "sqlglot==27.0.0",
    "pyarrow==19.0.1",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", *packages]
)
print("Required packages are available.")


print("\n=== Notebook 00 stage 2/7: configure pilot/full run ===")
from pathlib import Path
import hashlib
import json
import os
import sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# USER SETTING 1: use "pilot" first, then change to "full" and rerun.
RUN_MODE = "full"  # valid: pilot, full, extended

# USER SETTING 2: keep Drive enabled so outputs survive runtime resets.
USE_GOOGLE_DRIVE = True
if IN_COLAB and USE_GOOGLE_DRIVE:
    drive.mount("/content/drive")
    default_root = "/content/drive/MyDrive/Arbiter_ICDE_2027"
else:
    default_root = "/content/Arbiter_ICDE_2027"

PROJECT_ROOT = Path(
    os.environ.get("ARBITER_PROJECT_ROOT", default_root)
)
CODE_DIR = PROJECT_ROOT / "code"
SUITE_VERSION = "2026-08-01-audited-v4"
RUN_DIRECTORY = f"{RUN_MODE}_audited_v4"
RUN_DIR = PROJECT_ROOT / "runs" / RUN_DIRECTORY
OUTPUT_DIR = RUN_DIR / "00_workload"
import shutil
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
for directory in [CODE_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

mode_config = {
    "pilot": {
        "tpch_scale_factor": 0.1,
        "max_assets_per_dataset": 512,
        "quantile_count": 17,
        "semantic_checks_per_dataset": 12,
        "seeds": [11],
        "attack_budgets": [0.25],
        "attack_cases_per_relation_type": 4,
        "minimum_attack_cases_per_relation_type": 4,
        "scale_attack_cases_per_relation_type": 2,
        "scale_points": [128, 256, 512],
        "timing_repetitions": 3,
        "timing_warmups": 1,
        "concurrency_trials": 20,
    },
    "full": {
        "tpch_scale_factor": 1.0,
        "max_assets_per_dataset": 4096,
        "quantile_count": 33,
        "semantic_checks_per_dataset": 36,
        "seeds": [11, 23, 37, 51, 73],
        "attack_budgets": [0.10, 0.25, 0.50],
        "attack_cases_per_relation_type": 80,
        "minimum_attack_cases_per_relation_type": 20,
        "scale_attack_cases_per_relation_type": 4,
        "scale_points": [128, 512, 1024, 2048, 4096],
        "timing_repetitions": 10,
        "timing_warmups": 3,
        "concurrency_trials": 100,
    },
    "extended": {
        "tpch_scale_factor": 10.0,
        "max_assets_per_dataset": 8192,
        "quantile_count": 49,
        "semantic_checks_per_dataset": 48,
        "seeds": [11, 23, 37, 51, 73],
        "attack_budgets": [0.10, 0.25, 0.50],
        "attack_cases_per_relation_type": 80,
        "minimum_attack_cases_per_relation_type": 20,
        "scale_attack_cases_per_relation_type": 4,
        "scale_points": [128, 512, 1024, 2048, 4096, 8192],
        "timing_repetitions": 10,
        "timing_warmups": 3,
        "concurrency_trials": 200,
    },
}
if RUN_MODE not in mode_config:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")

config = {
    "run_mode": RUN_MODE,
    "suite_version": SUITE_VERSION,
    **mode_config[RUN_MODE],
    "tolerance": 1e-7,
    "repair_sweeps": 80,
    "repair_active_row_budget": 12000,
    "repair_step": 1.0,
    "osqp_max_iter": 20000,
}
(RUN_DIR / "run_config.json").write_text(
    json.dumps(config, indent=2, sort_keys=True),
    encoding="utf-8",
)
(PROJECT_ROOT / "active_run.json").write_text(
    json.dumps(
        {
            "run_mode": RUN_MODE,
            "run_directory": f"runs/{RUN_DIRECTORY}",
            "suite_version": SUITE_VERSION,
        },
        indent=2,
    ),
    encoding="utf-8",
)
print(json.dumps(config, indent=2))
print("Project root:", PROJECT_ROOT)


print("\n=== Notebook 00 stage 3/7: materialize and verify Arbiter modules ===")
import base64
import hashlib
import importlib
import sys

embedded_modules = {"arbiter_core.py": "...", "arbiter_data.py": "...", "arbiter_experiments.py": "...", "arbiter_workload.py": "..."}

# Import from Colab's local filesystem, which avoids Drive/FUSE import-cache
# edge cases.  Also persist the same bytes to Drive for Notebooks 01--04.
LOCAL_CODE_DIR = (
    Path(os.environ.get("ARBITER_LOCAL_CODE_ROOT", "/content/arbiter_runtime/code"))
    if IN_COLAB
    else CODE_DIR
)
for module_directory in {LOCAL_CODE_DIR, CODE_DIR}:
    module_directory.mkdir(parents=True, exist_ok=True)
    for name, encoded in embedded_modules.items():
        destination = module_directory / name
        destination.write_bytes(base64.b64decode(encoded))
module_hashes = {
    name: hashlib.sha256(base64.b64decode(encoded)).hexdigest()
    for name, encoded in embedded_modules.items()
}
(CODE_DIR / "code_manifest.json").write_text(
    json.dumps(module_hashes, indent=2, sort_keys=True),
    encoding="utf-8",
)

# Notebook 00 runs from Colab's /content directory, while the shared modules
# live in the persistent project directory on Google Drive.  Add that exact
# directory before importing any Arbiter module; local validation from the
# source folder would otherwise mask this Colab-only import failure.
code_dir_text = str(LOCAL_CODE_DIR.resolve())
if code_dir_text not in sys.path:
    sys.path.insert(0, code_dir_text)
importlib.invalidate_caches()

missing_modules = [name for name in embedded_modules if not (LOCAL_CODE_DIR / name).is_file()]
if missing_modules:
    raise FileNotFoundError(
        "Failed to materialize shared modules: " + ", ".join(missing_modules)
    )
print("Materialized shared source modules:")
for path in sorted(LOCAL_CODE_DIR.glob("arbiter_*.py")):
    print(" -", path.name, path.stat().st_size, "bytes")
print("Shared module path is active:", code_dir_text)

# Fail here with a precise diagnostic, before any experiment cell is reached.
for module_name in ["arbiter_core", "arbiter_workload", "arbiter_experiments", "arbiter_data"]:
    sys.modules.pop(module_name, None)
loaded_data_module = importlib.import_module("arbiter_data")
loaded_parent = Path(loaded_data_module.__file__).resolve().parent
if loaded_parent != LOCAL_CODE_DIR.resolve():
    raise ImportError(
        f"arbiter_data loaded from unexpected location: {loaded_data_module.__file__}"
    )
print("Startup import self-check PASS:", loaded_data_module.__file__)


print("\n=== Notebook 00 stage 4/7: prepare public/benchmark datasets ===")
# Leave these as None unless the automatic download fails.
MANUAL_NYC_PARQUET = None
MANUAL_ADULT_ZIP = None
FORCE_REBUILD_DATABASE = False

from arbiter_data import prepare_duckdb

connection, dataset_metadata = prepare_duckdb(
    project_root=PROJECT_ROOT,
    tpch_scale_factor=config["tpch_scale_factor"],
    force_rebuild=FORCE_REBUILD_DATABASE,
    nyc_parquet_path=MANUAL_NYC_PARQUET,
    adult_zip_path=MANUAL_ADULT_ZIP,
)
dataset_metadata.to_csv(
    OUTPUT_DIR / "dataset_metadata.csv", index=False
)
display(dataset_metadata)


print("\n=== Notebook 00 stage 5/7: generate and compile deterministic SQL workloads ===")
from arbiter_data import default_dataset_specs, save_json
from arbiter_workload import (
    compile_relations,
    evaluate_compiler,
    generate_deterministic_workload,
    oracle_relations,
    relation_sql_equivalence_checks,
)
import pandas as pd
import time

table_specs = default_dataset_specs(
    connection,
    tpch_scale_factor=config["tpch_scale_factor"],
    quantile_count=config["quantile_count"],
)
save_json(OUTPUT_DIR / "table_specs.json", table_specs)

asset_frames = []
for dataset, specs in table_specs.items():
    frame = generate_deterministic_workload(
        dataset=dataset,
        table_specs=specs,
        max_assets=config["max_assets_per_dataset"],
        append_negative_queries=True,
    )
    asset_frames.append(frame)
    print(
        dataset,
        int(frame["expected_supported"].sum()),
        "supported assets +",
        int((~frame["expected_supported"]).sum()),
        "negative parser cases",
    )
assets = pd.concat(asset_frames, ignore_index=True)

ground_truth = oracle_relations(assets)
compiled_frames = []
parse_frames = []
unsupported_frames = []
compiler_timing_rows = []
for dataset, dataset_assets in assets.groupby(
    "dataset", sort=False
):
    started = time.perf_counter_ns()
    dataset_compiled, dataset_parse, dataset_unsupported = (
        compile_relations(dataset_assets)
    )
    compile_ms = (time.perf_counter_ns() - started) / 1e6
    compiled_frames.append(dataset_compiled)
    parse_frames.append(dataset_parse)
    unsupported_frames.append(dataset_unsupported)
    compiler_timing_rows.append(
        {
            "dataset": dataset,
            "assets": len(dataset_assets),
            "supported_assets": int(
                dataset_assets["expected_supported"].sum()
            ),
            "compiled_relations": len(dataset_compiled),
            "sql_parse_and_relation_compile_ms": compile_ms,
        }
    )
compiled = pd.concat(compiled_frames, ignore_index=True)
parse_audit = pd.concat(parse_frames, ignore_index=True)
unsupported = pd.concat(unsupported_frames, ignore_index=True)
compiler_timing = pd.DataFrame(compiler_timing_rows)
compiler_metrics, relation_audit = evaluate_compiler(
    assets, ground_truth, compiled, parse_audit
)

display(compiler_metrics)
display(
    compiled.groupby(["dataset", "relation_type"])
    .size()
    .rename("relations")
    .reset_index()
)


print("\n=== Notebook 00 stage 6/7: execute semantic and plan checks ===")
from arbiter_data import extract_plan_features

plan_features = extract_plan_features(connection, assets)
assets = assets.merge(
    plan_features,
    on=["dataset", "asset_id"],
    how="left",
    validate="one_to_one",
)
semantic_checks = relation_sql_equivalence_checks(
    connection=connection,
    assets=assets,
    relations=compiled,
    max_relations_per_dataset=config[
        "semantic_checks_per_dataset"
    ],
)

assets.to_csv(OUTPUT_DIR / "sql_assets.csv", index=False)
ground_truth.to_csv(
    OUTPUT_DIR / "ground_truth_relations.csv", index=False
)
compiled.to_csv(OUTPUT_DIR / "compiled_relations.csv", index=False)
parse_audit.to_csv(OUTPUT_DIR / "query_parse_audit.csv", index=False)
unsupported.to_csv(
    OUTPUT_DIR / "unsupported_queries.csv", index=False
)
compiler_timing.to_csv(
    OUTPUT_DIR / "compiler_timing.csv", index=False
)
compiler_metrics.to_csv(
    OUTPUT_DIR / "compiler_metrics.csv", index=False
)
relation_audit.to_csv(
    OUTPUT_DIR / "relation_compilation_audit.csv", index=False
)
semantic_checks.to_csv(
    OUTPUT_DIR / "semantic_execution_checks.csv", index=False
)
display(
    semantic_checks.groupby(["dataset", "relation_type"])
    .agg(checks=("passed", "size"), pass_rate=("passed", "mean"))
    .reset_index()
)


print("\n=== Notebook 00 stage 7/7: write quality audit ===")
from arbiter_core import environment_manifest
from arbiter_data import save_json
import arbiter_core
import importlib.metadata

query_classification_ok = bool(
    (compiler_metrics["query_classification_accuracy"] == 1.0).all()
)
relation_precision_ok = bool(
    (compiler_metrics["precision"] == 1.0).all()
)
relation_recall_ok = bool(
    (compiler_metrics["recall"] == 1.0).all()
)
semantic_checks_ok = bool(
    len(semantic_checks) > 0 and semantic_checks["passed"].all()
)
plan_errors = int((assets["plan_status"] != "ok").sum())
pinned_versions_ok = bool(
    importlib.metadata.version("duckdb") == "1.3.2"
    and importlib.metadata.version("osqp") == "1.0.4"
    and importlib.metadata.version("sqlglot") == "27.0.0"
    and importlib.metadata.version("pyarrow") == "19.0.1"
)
expected_relation_types = {
    "decomposition",
    "projection_derivation",
    "filter_derivation",
}
relation_family_ok = bool(
    all(
        set(group["relation_type"]) == expected_relation_types
        for _, group in compiled.groupby("dataset")
    )
)
semantic_family_ok = bool(
    all(
        set(group["relation_type"]) == expected_relation_types
        for _, group in semantic_checks.groupby("dataset")
    )
)
quality = pd.DataFrame(
    [
        {
            "check": "exact parser/solver dependency versions",
            "passed": pinned_versions_ok,
        },
        {
            "check": "pinned OSQP backend is available",
            "passed": bool(arbiter_core.osqp is not None),
        },
        {
            "check": "supported/unsupported classification",
            "passed": query_classification_ok,
        },
        {
            "check": "compiler precision against construction oracle",
            "passed": relation_precision_ok,
        },
        {
            "check": "compiler recall against construction oracle",
            "passed": relation_recall_ok,
        },
        {
            "check": "all relation families exist per dataset",
            "passed": relation_family_ok,
        },
        {
            "check": "executed bag-semantics relation sample",
            "passed": semantic_checks_ok,
        },
        {
            "check": "semantic sample covers every relation family",
            "passed": semantic_family_ok,
        },
        {
            "check": "all assets compile in DuckDB EXPLAIN",
            "passed": plan_errors == 0,
        },
    ]
)
quality.to_csv(OUTPUT_DIR / "stage_quality_checks.csv", index=False)
save_json(
    OUTPUT_DIR / "environment.json",
    environment_manifest(config),
)
display(quality)
if not quality["passed"].all():
    raise RuntimeError(
        "Notebook 00 failed one or more quality checks. "
        "Do not run later notebooks until the failed rows are resolved."
    )
print("Notebook 00 PASS. Continue to Notebook 01.")



=== Notebook 00 stage 1/7: install dependencies ===
Required packages are available.

=== Notebook 00 stage 2/7: configure pilot/full run ===
Mounted at /content/drive
{
  "run_mode": "full",
  "suite_version": "2026-08-01-audited-v4",
  "tpch_scale_factor": 1.0,
  "max_assets_per_dataset": 4096,
  "quantile_count": 33,
  "semantic_checks_per_dataset": 36,
  "seeds": [
    11,
    23,
    37,
    51,
    73
  ],
  "attack_budgets": [
    0.1,
    0.25,
    0.5
  ],
  "attack_cases_per_relation_type": 80,
  "minimum_attack_cases_per_relation_type": 20,
  "scale_attack_cases_per_relation_type": 4,
  "scale_points": [
    128,
    512,
    1024,
    2048,
    4096
  ],
  "timing_repetitions": 10,
  "timing_warmups": 3,
  "concurrency_trials": 100,
  "tolerance": 1e-07,
  "repair_sweeps": 80,
  "repair_active_row_budget": 12000,
  "repair_step": 1.0,
  "osqp_max_iter": 20000
}
Project root: /content/drive/MyDrive/Arbiter_ICDE_2027

=== Notebook 00 stage 3/7: materialize and verify Arbiter

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,dataset,provenance,source_url,local_file,sha256,total_rows,materialization
0,NYC Taxi 2024-01,public observational,https://d37ci6vzurychx.cloudfront.net/trip-dat...,/content/drive/MyDrive/Arbiter_ICDE_2027/input...,c4d59da7bbc8abaeeeb1727947ee93d9891a71acb42854...,2964624,NaN
1,TPC-H SF1,benchmark-generated,https://www.tpc.org/tpch/,/content/drive/MyDrive/Arbiter_ICDE_2027/state...,,8661245,cached
2,UCI Adult,public observational,https://archive.ics.uci.edu/static/public/2/ad...,/content/drive/MyDrive/Arbiter_ICDE_2027/input...,7537312dd56c2b98035880805ce99e68183a30ee468aa5...,48842,NaN



=== Notebook 00 stage 5/7: generate and compile deterministic SQL workloads ===


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,dataset,relation_type,checks,pass_rate
0,NYC Taxi 2024-01,decomposition,12,1.0
1,NYC Taxi 2024-01,filter_derivation,12,1.0
2,NYC Taxi 2024-01,projection_derivation,12,1.0
3,TPC-H SF1,decomposition,12,1.0
4,TPC-H SF1,filter_derivation,12,1.0
5,TPC-H SF1,projection_derivation,12,1.0
6,UCI Adult,decomposition,12,1.0
7,UCI Adult,filter_derivation,12,1.0
8,UCI Adult,projection_derivation,12,1.0



=== Notebook 00 stage 7/7: write quality audit ===


,check,passed
0,exact parser/solver dependency versions,True
1,pinned OSQP backend is available,True
2,supported/unsupported classification,True
3,compiler precision against construction oracle,True
4,compiler recall against construction oracle,True
5,all relation families exist per dataset,True
6,executed bag-semantics relation sample,True
7,semantic sample covers every relation family,True
8,all assets compile in DuckDB EXPLAIN,True


Notebook 00 PASS. Continue to Notebook 01.
